In [1]:
%pwd

'f:\\end_to_end_AI_Projects\\Medical-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS\\research'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'f:\\end_to_end_AI_Projects\\Medical-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS'

In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

f:\Apps\Anaconda\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Extract text from PDF files
def load_pdf_files(file_path):
    loader = DirectoryLoader(file_path, glob="*.pdf", show_progress=True, loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [6]:
extracted_data = load_pdf_files('data/')

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:38<00:00, 38.56s/it]


In [8]:
extracted_data[0]

Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf', 'total_pages': 637, 'page': 0, 'page_label': '1'}, page_content='')

In [9]:
len(extracted_data)

637

In [7]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:

    """
    Given a list of documents objects, return a new list of document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [8]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [9]:
minimal_docs[0]

Document(metadata={'source': 'data\\Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf'}, page_content='')

In [10]:
# split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20, length_function=len)
    texts_chunks = text_splitter.split_documents(minimal_docs)
    return texts_chunks

In [11]:
texts_chunks = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunks)}")

Number of chunks: 5859


In [12]:
from langchain.embeddings import HuggingFaceBgeEmbeddings
import torch
def download_embeddings():
    """
    Download and return the huggingface embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceBgeEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"}
        )
    return embeddings

embeddings = download_embeddings()


C:\Users\AL-MASA\AppData\Local\Temp\ipykernel_14772\3670647268.py:8: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(


In [16]:
vector=embeddings.embed_query("What is the capital of France?")
vector

[0.08866880089044571,
 0.08471289277076721,
 0.021126747131347656,
 -0.026203695684671402,
 0.046463508158922195,
 0.007219620048999786,
 0.032416995614767075,
 0.028424998745322227,
 -0.012244842946529388,
 -0.04541474208235741,
 -0.0015061245067045093,
 -0.11714604496955872,
 0.0334186889231205,
 -0.05487793684005737,
 -0.024233050644397736,
 -0.09996766597032547,
 0.0015219508204609156,
 -0.018644683063030243,
 0.03697919100522995,
 0.0028686262667179108,
 0.09007009863853455,
 -0.005486339330673218,
 0.05808258429169655,
 -0.022710032761096954,
 0.00958606693893671,
 -0.04708489030599594,
 -0.005326659884303808,
 -0.012992486357688904,
 0.02711320109665394,
 -0.01247719582170248,
 0.0326518788933754,
 -0.01711278222501278,
 -0.012226824648678303,
 -0.01657683216035366,
 -0.004478172864764929,
 -0.018467001616954803,
 0.00013797165593132377,
 -0.011391846463084221,
 0.10035791993141174,
 0.021818649023771286,
 -0.01909705623984337,
 -0.06312574446201324,
 -0.04365519806742668,
 -0.0

In [17]:
print("vector length:", len(vector))

vector length: 384


In [13]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [14]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [15]:
from pinecone import Pinecone
pinecone_api_key=PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [16]:
pc

In [17]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [23]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunks,
    embedding=embeddings,
    index_name=index_name
)

In [18]:
# Load Existing index

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

### Add more data to the existing pinecone index

In [19]:
dswith = Document(
    page_content="What is the capital of France?",
    metadata={"source": "test_source"}
)

In [20]:
docsearch.add_documents(documents=[dswith])

['4268caf9-f50d-4876-acf3-5a25b7f6b4a9']

In [21]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [22]:
retrieved_docs = retriever.invoke("What is the capital of France?")
retrieved_docs

[Document(id='2892271e-ca0e-4a01-be78-ac4f3220a180', metadata={'source': 'test_source'}, page_content='What is the capital of France?'),
 Document(id='60a2ed3c-4c19-43df-9cd0-e55d5ce1e48b', metadata={'source': 'test_source'}, page_content='What is the capital of France?'),
 Document(id='2f3e21c1-6d0f-4121-88a9-a9d08a85d54f', metadata={'source': 'test_source'}, page_content='What is the capital of France?')]

In [32]:
from langchain_groq import ChatGroq

chat_groq = ChatGroq(model="openai/gpt-oss-20b")

In [33]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [34]:
system_prompt = (
    "you are as medical assistant for question-answering tasks. "
    "use the following retrieved documents to answer the question. "
    "the question. If you don't know the answer, say you don't know. "
    "don't know. Use three sentences at max to answer the question. "
    "answer concise."
    "\n\n"
    "{context}"

)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [35]:
question_answering_chain = create_stuff_documents_chain(chat_groq, prompt)
rag_chain = create_retrieval_chain(retriever, question_answering_chain)

In [36]:
response = rag_chain.invoke({"input":"what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly and gigantism are disorders caused by excess growth hormone released from the pituitary gland.  
Acromegaly develops after the growth plates have closed, leading to enlarged hands, feet, and facial features, while gigantism occurs before closure, resulting in abnormal height.  
Both conditions cause a range of systemic disturbances beyond bone and soft‑tissue overgrowth.


In [37]:
response = rag_chain.invoke({"input":"what is Acne?"})
print(response["answer"])

Acne is a common inflammatory skin disorder that mainly affects the face, chest, and back. It is characterized by clogged hair follicles (comedones) and inflammatory lesions such as papules, pustules, and sometimes cysts. The condition results from follicular hyperkeratinization, excess sebum production, bacterial colonization, and an inflammatory response.


In [38]:
response = rag_chain.invoke({"input":"what is the treatment of Acne?"})
print(response["answer"])

Acne is usually treated with topical agents such as benzoyl peroxide, salicylic acid, or retinoids to reduce inflammation and unclog pores.  For moderate to severe cases, oral antibiotics or hormonal therapies (e.g., oral contraceptives) may be prescribed, and isotretinoin is reserved for recalcitrant disease.  Treatment is often individualized and may combine multiple modalities for best results.
